# Spam Email Classification Using Machine Learning


## 1. Install and import libraries

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc
)

from imblearn.over_sampling import SMOTE

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

## 2. Load the dataset

In [ ]:
# Upload SpamAssasin.csv to the Colab session, or update this path.
df = pd.read_csv("SpamAssasin.csv")
display(df.head())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

## 3. Handle missing values and inspect the data

In [ ]:
required_columns = ["sender", "receiver", "subject", "body", "date", "label"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["receiver"] = df["receiver"].fillna("unknown")
df["sender"] = df["sender"].fillna("unknown")
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")
df["date"] = df["date"].fillna("")

print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
print(df.isnull().sum())
print("\nLabel distribution:")
print(df["label"].value_counts())

## 4. Feature engineering

In [ ]:
def extract_domain(email):
    if isinstance(email, str) and "@" in email:
        match = re.search(r"@([a-zA-Z0-9.-]+)", email)
        return match.group(1).lower() if match else "unknown"
    return "unknown"

df["sender_domain"] = df["sender"].apply(extract_domain)
df["receiver_domain"] = df["receiver"].apply(extract_domain)

df["date"] = pd.to_datetime(
    df["date"], format="%m/%d/%y%H:%M", errors="coerce"
)
df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["weekday"] = df["date"].dt.weekday

def count_urls(text):
    pattern = r"https?://[^\s]+"
    return len(re.findall(pattern, str(text)))

df["url_count"] = (
    df["subject"].apply(count_urls) + df["body"].apply(count_urls)
)

display(df[["sender_domain", "receiver_domain", "day", "month",
            "year", "weekday", "url_count"]].head())

## 5. Encode labels and preprocess email text

In [ ]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["label"])

print("Label mapping:")
for original, encoded in zip(label_encoder.classes_,
                             label_encoder.transform(label_encoder.classes_)):
    print(f"{original} -> {encoded}")

In [ ]:
# Clean text and combine subject and body.
def clean_text(text):
    text = re.sub(r"[^a-zA-Z\s]", " ", str(text))
    text = text.lower()
    return re.sub(r"\s+", " ", text).strip()

df["subject_clean"] = df["subject"].apply(clean_text)
df["body_clean"] = df["body"].apply(clean_text)
df["email_text"] = (df["subject_clean"] + " " + df["body_clean"]).str.strip()

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    return " ".join(tokens)

df["tokens"] = df["email_text"].apply(preprocess_text)
display(df[["email_text", "tokens", "label"]].head())

## 6. Split the data and extract TF-IDF features



In [ ]:
X_text = df["tokens"]
y = df["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train = tfidf_vectorizer.fit_transform(X_train_text)
X_test = tfidf_vectorizer.transform(X_test_text)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

## 7. Handle class imbalance using SMOTE

In [ ]:
print("Training labels before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train, y_train
)

print("\nTraining labels after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

## 8. Train five classification models



In [ ]:
base_learners = [
    ("log_reg", LogisticRegression(max_iter=1000, random_state=42)),
    ("nb", MultinomialNB()),
    ("rf", RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    )),
    ("xgb", XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ))
]

models = {
    "Logistic Regression": base_learners[0][1],
    "Naive Bayes": base_learners[1][1],
    "Random Forest": base_learners[2][1],
    "XGBoost": base_learners[3][1],
    "Stacking Classifier": StackingClassifier(
        estimators=base_learners,
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5,
        n_jobs=-1
    )
}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_resampled, y_train_resampled)

print("All five models trained successfully.")

## 9. Evaluate the models

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Sensitivity (Recall)": recall_score(y_test, y_pred, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else 0,
        "F1 Score": f1_score(y_test, y_pred, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values(
    by="Accuracy", ascending=False
)
display(results_df.round(4))

## 10. Classification reports

In [ ]:
for name, y_pred in predictions.items():
    print(f"\n{name}")
    print(classification_report(
        y_test, y_pred, zero_division=0,
        target_names=[str(x) for x in label_encoder.classes_]
    ))

## 11. Confusion matrices

In [ ]:
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Not Spam", "Spam"],
        yticklabels=["Not Spam", "Spam"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.show()

## 12. Compare model performance

In [ ]:
metric_columns = [
    "Accuracy", "Precision", "Sensitivity (Recall)",
    "Specificity", "F1 Score"
]

results_df.set_index("Model")[metric_columns].plot(
    kind="bar", figsize=(12, 6)
)
plt.title("Performance Comparison of Five Classifiers")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 13. Test with new email samples

In [ ]:
def predict_email(email, trained_model=models["Stacking Classifier"]):
    cleaned = clean_text(email)
    processed = preprocess_text(cleaned)
    features = tfidf_vectorizer.transform([processed])
    prediction = trained_model.predict(features)[0]
    return label_encoder.inverse_transform([prediction])[0]

sample_emails = [
    "Limited offer! Get 70% off on all products. Click here!",
    "Lunch meeting confirmed for 1 PM today."
]

for email in sample_emails:
    print("Email:", email)
    print("Prediction:", predict_email(email))
    print("-" * 60)